# Premier League Match Analysis and Prediction

Analysing four Premier League seasons (2021/22 to 2024/25) with SQL, then predicting match results with a Poisson model and comparing those predictions against Bet365's odds.

## Step 0: Setup check

This cell imports every library the project needs and prints its version. If it runs without an error, the environment is ready.

In [1]:
import sys                # Tells us which Python is running this notebook
import sqlite3            # SQL database, built into Python (no install needed)

import pandas as pd       # Tables of data (loading, cleaning, grouping)
import numpy as np        # Fast maths on arrays of numbers
import scipy              # Scientific maths (we'll use its Poisson distribution)
import matplotlib         # Charts

# This path should point inside the project's venv folder.
# If it does, the notebook is using the right environment.
print("Python running from:", sys.executable)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("sqlite:", sqlite3.sqlite_version)

Python running from: D:\Desktop\premier-league-match-predictor\venv\Scripts\python.exe
pandas: 3.1.1
numpy: 2.5.4
scipy: 1.18.1
matplotlib: 3.11.2
sqlite: 3.50.4


## Step 1: Load the data

Each season is a separate CSV file on football-data.co.uk. This cell downloads all four, labels every row with its season, and stacks them into one table. A Premier League season has 20 teams playing 380 matches, so we expect 380 rows per season and 1,520 in total.

In [2]:
# Each season's URL uses a short code: "2122" means 2021/22.
# This dictionary pairs each code with the label we want to use for that season.
seasons = {
    "2122": "2021/22",
    "2223": "2022/23",
    "2324": "2023/24",
    "2425": "2024/25",
}

season_tables = []  # An empty list to hold one table per season

for code, label in seasons.items():
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"  # E0 = England's top division
    season_df = pd.read_csv(url).copy()  # Download the CSV into a table (.copy() tidies its memory layout)
    season_df["Season"] = label          # Add a column recording which season each row is from
    season_tables.append(season_df)      # Keep it until all four are loaded

# Stack the four tables on top of each other. ignore_index=True renumbers the rows
# 0, 1, 2... so that row numbers don't restart at 0 for each season.
raw_matches = pd.concat(season_tables, ignore_index=True)

# Check 1: the overall size, as (rows, columns)
print("Rows and columns:", raw_matches.shape)

# Check 2: each season should have exactly 380 matches
print("\nMatches per season:")
print(raw_matches["Season"].value_counts().sort_index())

# Check 3: look at the first few rows, using only the columns we plan to keep
columns_to_show = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]
raw_matches[columns_to_show].head()

Rows and columns: (1520, 133)

Matches per season:
Season
2021/22    380
2022/23    380
2023/24    380
2024/25    380
Name: count, dtype: int64


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A
0,2021/22,13/08/2021,Brentford,Arsenal,2,0,H,4.00,3.40,1.95
1,2021/22,14/08/2021,Man United,Leeds,5,1,H,1.53,4.50,5.75
2,2021/22,14/08/2021,Burnley,Brighton,1,2,A,3.10,3.10,2.45
3,2021/22,14/08/2021,Chelsea,Crystal Palace,3,0,H,1.25,5.75,13.00
4,2021/22,14/08/2021,Everton,Southampton,3,1,H,1.90,3.50,4.00


## Step 2: Clean the data

The raw table has 133 columns, but this project only needs 10. The first cell keeps those columns, then checks for missing values and looks at each column's data type.

In [3]:
# The 10 columns the project needs
columns_to_keep = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]

# .copy() makes an independent table, so changes here can't affect raw_matches
matches = raw_matches[columns_to_keep].copy()

# isna() marks each empty cell as True, and sum() counts them for each column
print("Missing values in each column:")
print(matches.isna().sum())

# dtypes shows how pandas is storing each column
print("\nData types before cleaning:")
print(matches.dtypes)

Missing values in each column:
Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
B365H       0
B365D       0
B365A       0
dtype: int64

Data types before cleaning:
Season          str
Date            str
HomeTeam        str
AwayTeam        str
FTHG          int64
FTAG          int64
FTR             str
B365H       float64
B365D       float64
B365A       float64
dtype: object


In [4]:
# Date is stored as text, so convert it to a real date.
# The format is day/month/year. Stating it means 05/08/2022 is read as 5 August, not May 8.
matches["Date"] = pd.to_datetime(matches["Date"], format="%d/%m/%Y")

# Put the matches in time order. reset_index(drop=True) renumbers the rows afterwards.
matches = matches.sort_values("Date").reset_index(drop=True)

print("Data type of Date now:", matches["Date"].dtype)
print("\nFirst and last match date in each season:")
print(matches.groupby("Season")["Date"].agg(["min", "max"]))

Data type of Date now: datetime64[us]

First and last match date in each season:
               min        max
Season                       
2021/22 2021-08-13 2022-05-22
2022/23 2022-08-05 2023-05-28
2023/24 2023-08-11 2024-05-19
2024/25 2024-08-16 2025-05-25


In [5]:
# Sanity checks: no missing values doesn't guarantee that the values make sense.

# Check 1: each home/away pairing should happen only once per season
duplicates = matches.duplicated(subset=["Season", "HomeTeam", "AwayTeam"]).sum()
print("Duplicate fixtures:", duplicates)

# Check 2: work out the result from the score and compare it with the FTR column
calculated_result = np.where(matches["FTHG"] > matches["FTAG"], "H",
                    np.where(matches["FTHG"] < matches["FTAG"], "A", "D"))
print("Results that don't match the score:", (calculated_result != matches["FTR"]).sum())

# Check 3: goals can't be negative, and decimal odds must be above 1
print("Lowest goal count:", matches[["FTHG", "FTAG"]].min().min())
print("Lowest odds:", matches[["B365H", "B365D", "B365A"]].min().min())

# Check 4: every season should have 20 teams
print("\nTeams per season:")
print(matches.groupby("Season")["HomeTeam"].nunique())

# Check 5: read the team names, looking for the same club spelled two ways
all_teams = sorted(set(matches["HomeTeam"]) | set(matches["AwayTeam"]))
print("\nAll", len(all_teams), "teams:")
print(all_teams)

print("\nFinal shape:", matches.shape)

Duplicate fixtures: 0
Results that don't match the score: 0
Lowest goal count: 0
Lowest odds: 1.06

Teams per season:
Season
2021/22    20
2022/23    20
2023/24    20
2024/25    20
Name: HomeTeam, dtype: int64

All 26 teams:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Tottenham', 'Watford', 'West Ham', 'Wolves']

Final shape: (1520, 10)


### Cleaning decisions

| Decision | Reason |
|---|---|
| Kept 10 of the 133 columns | The project only uses the match details, the score, the result and Bet365's odds. |
| Used Bet365 as the only bookmaker | One bookmaker keeps the comparison simple, and its odds are complete for all 1,520 matches. |
| Removed no rows | There are no missing values or duplicate fixtures, so there is nothing to remove. Each season keeps all 380 matches. |
| Converted `Date` from text to a real date | Dates stored as text can't be reliably sorted or compared. The format is stated explicitly so that day and month can't be swapped. |
| Sorted the matches by date | The model must only learn from the past, so the data should be in time order. |
| Left the other data types unchanged | Goals are already whole numbers, odds are already decimals, and team names, results and seasons are already text. |
| Left team names unchanged | There are 26 different clubs across the four seasons, and each club has one spelling. |
| Kept high-scoring matches and long odds | The highest values (9 goals, odds of 29.0) are unusual but realistic. Removing them would hide real results from the model. |